# 🔬 Comprehensive Hyperparameter Tuning: CoNLL-2003

**8 Systematic Experiments** exploring olfactory-NER hyperparameters:

1. Baseline (ReLU)
2. Baseline (GELU) 
3. More Receptors (256)
4. More Glomeruli (64)
5. Larger LSTM (512)
6. Lower Dropout (0.2)
7. Larger Batch (64)
8. Strong Regularization

**Each experiment auto**:
- ✅ Trains & evaluates
- ✅ Saves model to Google Drive
- ✅ Saves results to GitHub
- ✅ Generates visualizations

## Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!git clone https://github.com/bhushan1729/olfaction-inspired-ner.git
%cd olfaction-inspired-ner

In [ ]:
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard

In [ ]:
import os
if not os.path.exists('./data/glove.6B.300d.txt'):
    !mkdir -p data
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip
    !unzip -q data/glove.6B.zip -d data/
    !rm data/glove.6B.zip

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model_dir = '/content/drive/MyDrive/olfaction_ner/models'
!mkdir -p {model_dir}
print(f"Models→ {model_dir}")

## Helper Functions

In [ ]:
import json, shutil, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

!mkdir -p comparison_plots experiment_results/CoNLL-2003
all_experiments = []

def save_experiment_to_drive(name, save_dir):
    dst = f'/content/drive/MyDrive/olfaction_ner/models/{name}_best.pt'
    shutil.copy(f"{save_dir}/best_model.pt", dst)
    print(f"✓ Saved to Drive: {dst}")

def save_results(name, config, results, save_dir):
    exp_dir = f"experiment_results/CoNLL-2003/{name}"
    Path(exp_dir).mkdir(parents=True, exist_ok=True)
    
    with open(f"{exp_dir}/metadata.json", 'w') as f:
        json.dump({'experiment_name': name, 'config': config}, f, indent=2)
    with open(f"{exp_dir}/results.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    all_experiments.append({'name': name, 'config': config, 'results': results})
    print(f"✓ Saved results: {exp_dir}")

def plot_curves(name, results, path):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    epochs = [e['epoch'] for e in results['epochs']]
    ax[0].plot(epochs, [e['train']['total_loss'] for e in results['epochs']], 'o-')
    ax[0].set_title(f'{name}: Training Loss')
    ax[0].grid(True, alpha=0.3)
    
    ax[1].plot(epochs, [e['valid']['f1'] for e in results['epochs']], 'o-', color='green')
    ax[1].set_title(f'{name}: Validation F1')
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_entities(name, results, path):
    per_entity = results['test']['per_entity']
    entities = [k for k in per_entity if k not in ['micro avg', 'macro avg', 'weighted avg']]
    
    plt.figure(figsize=(10, 5))
    plt.bar(entities, [per_entity[e] for e in entities], color='steelblue', alpha=0.8)
    plt.title(f'{name}: Per-Entity F1')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()

print("✓ Ready")

---
# Experiments
---